# Step 7 — First analysis: uncertainty and Treasury reactions
**Run Runtime → Run all**, then save as **06_initial_analysis.ipynb** in ResearchProject / main. The visible setup repeats the validated data build so this notebook works in a fresh Colab runtime. Allow several minutes for downloads and bootstrap draws.

**Primary question:** does pre-announcement Kalshi decision uncertainty explain absolute two-year Treasury statement-window moves, and does it add information beyond six-month conventional policy-path uncertainty? Revision volatility is a separate planned predictor comparison.

**Fixed models:** A: outcome ~ sigma. B0/B1: outcome ~ baseline versus outcome ~ baseline + sigma, on identical rows. R0/R1: baseline versus baseline + RV, estimated in parallel. All models have an intercept. Main tests are two-sided. No kitchen-sink model or predictor selection based on p-values.

**Interpretation:** coefficients are yield-movement bps per predictor bp. HC3 intervals address unequal residual variability; they do not fix measurement errors or guarantee small-sample validity. Restricted wild bootstrap provides an approximate small-sample sensitivity check. Leave-one-event-out estimates show dependence on individual meetings. This is historical in-sample analysis; no causal or demonstrated forecasting claim is supported.

**Known limits:** 37 candidate events; authors' processed and sometimes forward-filled prices; five constructed zero-sigma observations; six zero-volume-day revision windows; current-vintage rather than verified real-time data. UTC daily timing replaces the original 16:00 Eastern cutoff. A six-month benchmark measures a different horizon from the immediate decision.

**Sources:** [Kalshi replication package](https://github.com/jdkatz21/Prediction_Markets_Public), [USMPD](https://www.frbsf.org/research-and-insights/data-and-indicators/us-monetary-policy-event-study-database/), [conventional uncertainty](https://www.frbsf.org/research-and-insights/data-and-indicators/market-based-monetary-policy-uncertainty/), [statsmodels OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html).

No realized policy surprise is used as an ex-ante predictor. Supporting actual-decision surprise and remaining measurement sensitivities are separate unfinished work.

## Verified initial run — October 19, 2026
All 14 code cells executed successfully in a fresh local scope and all three figures were inspected. Run this notebook in Colab to produce and save its outputs there.

For the current 37-event sample, the adjusted sigma coefficient is **0.403** (95% HC3 interval **0.090 to 0.716**). Adding sigma raises in-sample R² from **0.097 to 0.223**. Its approximate wild-bootstrap p-value is **0.0116** (**0.0232** after Holm adjustment across the two incremental predictor tests). Single-event omissions leave the sigma coefficient between **0.337 and 0.462**.

The revision-RV coefficient is much less precise: **0.357**, with 95% HC3 interval **−0.671 to 1.385**. The sigma relationship also weakens in the broader statement-plus-press-conference window: coefficient **0.213**, interval **−0.285 to 0.711**. Thus the initial evidence concerns the narrow statement window and does not establish a general forecasting result. Measurement and source-construction sensitivities remain unfinished.

These numbers document this dated run; later source updates may change outputs. Do not treat this snapshot as a pre-registered or out-of-sample result.

### Setup 1: Download published sources

In [ ]:
import io
import json
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
import numpy as np
import pandas as pd

# Fixed scope. Refreshing the files can update history; hashes identify each download.
STUDY_START = pd.Timestamp("2022-01-01")
AS_OF_DATE = pd.Timestamp.now(tz="America/New_York").tz_localize(None).normalize()
SOURCES = {
  "moments": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_moments_data/daily_moments_fed_levels.csv",
  "distributions": "https://kalshi-and-the-rise-of-macro-markets.s3.amazonaws.com/daily_distribution_data/daily_distributions_fed_levels.csv",
  "usmpd": "https://www.frbsf.org/wp-content/uploads/USMPD.xlsx"
}
source_receipts = []
source_bytes = {}

def download(name, url):
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "ResearchProject/1.0"})
            with urllib.request.urlopen(request, timeout=45) as response:
                data = response.read()
                modified = response.headers.get("Last-Modified")
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    source_bytes[name] = data
    source_receipts.append({
        "source": name, "url": url, "bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
        "retrieved_utc": pd.Timestamp.now(tz="UTC").isoformat(),
        "last_modified_http": modified
    })
    return data

moments = pd.read_csv(io.BytesIO(download("moments", SOURCES["moments"])))
distributions = pd.read_csv(io.BytesIO(download("distributions", SOURCES["distributions"])))
workbook = pd.ExcelFile(io.BytesIO(download("usmpd", SOURCES["usmpd"])))
statements = pd.read_excel(workbook, sheet_name="Statements")
print("Downloaded published moments, distributions and USMPD.")
print("No account credentials or API keys are needed.")


### Setup 2: Validate schemas and dates

In [ ]:
required_m = {"date", "contract_preamble", "mean", "variance", "daily_volume"}
required_d = {"date", "contract_preamble", "strike", "probability"}
assert required_m <= set(moments.columns), "Moments schema changed"
assert required_d <= set(distributions.columns), "Distribution schema changed"
assert {"Date", "Unscheduled", "UST2Y"} <= set(statements.columns), "USMPD schema changed"

for frame in [moments, distributions]:
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    assert frame["date"].notna().all()
    frame["event_key"] = frame["contract_preamble"].str.replace(r"^KX", "", regex=True)
assert not moments.duplicated(["event_key", "date"]).any(), "Duplicate event/date moments"
assert not distributions.duplicated(["event_key", "date", "strike"]).any(), "Duplicate bins"
for column in ["mean", "variance", "daily_volume"]:
    moments[column] = pd.to_numeric(moments[column], errors="raise")
for column in ["strike", "probability"]:
    distributions[column] = pd.to_numeric(distributions[column], errors="raise")
assert moments["variance"].dropna().ge(-1e-12).all(), "Negative variance"
assert distributions["probability"].dropna().between(0, 100 + 1e-8).all()

statements["Date"] = pd.to_datetime(statements["Date"], errors="raise")
assert statements["Date"].is_unique
assert statements["Unscheduled"].isin([0, 1]).all()
statements["UST2Y"] = pd.to_numeric(statements["UST2Y"], errors="raise")
# Never use the source's expiry_date as an official meeting date:
# the authors calculate it from the last observed trade day.
events = statements.loc[
    statements["Unscheduled"].eq(0) &
    statements["Date"].ge(STUDY_START) &
    statements["Date"].le(AS_OF_DATE), ["Date", "UST2Y"]
].rename(columns={"Date": "meeting_date", "UST2Y": "ust2y_change_pp"}).copy()
events["event_key"] = events["meeting_date"].map(
    lambda d: "FED-" + d.strftime("%y%b").upper()
)
assert events["event_key"].is_unique, "Two meetings share an event month; inspect mapping"
events["observation_date"] = events["meeting_date"] - pd.offsets.BDay(1)
assert events["observation_date"].lt(events["meeting_date"]).all()
print("Kalshi observation range:", moments["date"].min().date(), "to", moments["date"].max().date())
print("Latest USMPD statement:", statements["Date"].max().date())
print("Scheduled meetings in scope:", len(events))
print("Event mapping uses official USMPD meeting dates and month identifiers.")
print("Previous weekday convention: selected FOMC cutoffs must also be checked for holidays.")


### Setup 3: Reconcile published moments

In [ ]:
# Join exactly one published daily observation to each scheduled event.
audit = events.merge(
    moments[["event_key", "date", "mean", "variance", "daily_volume"]],
    left_on=["event_key", "observation_date"],
    right_on=["event_key", "date"], how="left", validate="one_to_one",
    indicator="moments_match"
)
assert len(audit) == len(events)
targets = events[["event_key", "observation_date"]]
bins = distributions.merge(
    targets, left_on=["event_key", "date"],
    right_on=["event_key", "observation_date"],
    how="inner", validate="many_to_one"
)
checks = []
for (event_key, date), group in bins.groupby(["event_key", "date"]):
    p = group["probability"].to_numpy(dtype=float) / 100
    x = group["strike"].to_numpy(dtype=float)
    finite = np.isfinite(p).all() and np.isfinite(x).all()
    total = p.sum()
    if finite and total > 0:
        weights = p / total
        mean_unadjusted = np.sum(weights * x)
        var_rebuilt = np.sum(weights * (x - mean_unadjusted) ** 2)
    else:
        mean_unadjusted = var_rebuilt = np.nan
    checks.append({
        "event_key": event_key, "observation_date": date,
        "probability_sum": total, "bins": len(group),
        "rebuilt_variance_pp2": var_rebuilt,
        "rebuilt_mean_pp": mean_unadjusted + 0.125,
        "distribution_valid": bool(finite and np.isclose(total, 1, atol=1e-8))
    })
checks = pd.DataFrame(checks, columns=[
    "event_key", "observation_date", "probability_sum", "bins",
    "rebuilt_variance_pp2", "rebuilt_mean_pp", "distribution_valid"
])
audit = audit.merge(checks, on=["event_key", "observation_date"],
                    how="left", validate="one_to_one")
audit["variance_matches"] = np.isclose(
    audit["variance"], audit["rebuilt_variance_pp2"], atol=1e-10, rtol=1e-7
)
audit["mean_matches"] = np.isclose(
    audit["mean"], audit["rebuilt_mean_pp"], atol=1e-8, rtol=1e-7
)
audit["ust2y_signed_bps"] = audit["ust2y_change_pp"] * 100
audit["abs_ust2y_bps"] = audit["ust2y_signed_bps"].abs()
audit["kalshi_sigma_bps"] = np.sqrt(audit["variance"].clip(lower=0)) * 100

def inclusion_reason(row):
    if row["moments_match"] != "both":
        return "exclude: missing exact previous-day moments"
    if not np.isfinite(row["ust2y_signed_bps"]):
        return "exclude: missing Treasury outcome"
    if not np.isfinite(row["variance"]) or row["variance"] < -1e-12:
        return "exclude: invalid variance"
    if not (pd.notna(row["distribution_valid"]) and row["distribution_valid"]):
        return "exclude: missing or invalid distribution"
    if not row["variance_matches"] or not row["mean_matches"]:
        return "exclude: published moments do not reconcile"
    return "include: published-method candidate"

audit["inclusion_reason"] = audit.apply(inclusion_reason, axis=1)
audit["included"] = audit["inclusion_reason"].eq("include: published-method candidate")
print(audit[["meeting_date", "observation_date", "kalshi_sigma_bps",
             "abs_ust2y_bps", "inclusion_reason"]].round(3).to_string(index=False))
print("\nYear-level coverage:")
print(audit.groupby(audit["meeting_date"].dt.year).agg(
    scheduled=("included", "size"), included=("included", "sum")
).to_string())


### Setup 4: Build the candidate dataset

In [ ]:
dataset = audit.loc[audit["included"], [
    "meeting_date", "event_key", "observation_date", "mean",
    "kalshi_sigma_bps", "ust2y_signed_bps", "abs_ust2y_bps",
    "daily_volume", "probability_sum", "bins"
]].rename(columns={
    "mean": "authors_adjusted_mean_rate_pct",
    "daily_volume": "authors_daily_volume"
}).sort_values("meeting_date").reset_index(drop=True)

assert dataset["meeting_date"].is_unique
assert dataset["observation_date"].lt(dataset["meeting_date"]).all()
assert np.isfinite(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]]).all().all()
assert dataset["kalshi_sigma_bps"].ge(0).all()
print("Reconciled candidate events:", len(dataset))
print("Excluded scheduled events:", int((~audit["included"]).sum()))
print("Zero constructed uncertainty:", int(dataset["kalshi_sigma_bps"].eq(0).sum()))
print("\nDescriptive ranges (basis points):")
print(dataset[["kalshi_sigma_bps", "abs_ust2y_bps"]].describe().round(3).to_string())
print("\nLIMITS:")
print("- This is a dataset using the authors' processed measures, not an exact raw distribution.")
print("- Daily prices can be carried forward; daily volume does not establish age of every strike.")
print("- UTC calendar-day aggregation replaces the original 16:00 Eastern cutoff.")
print("- Reconciled means include the authors' 0.125 percentage-point adjustment.")
print("- Constructed zero variance does not establish zero true uncertainty.")
print("- Corrections, tail conventions and retrospective processing require sensitivity checks.")
print("- No claims of predictive skill or causality are made; no regressions have been run.")
print("- Policy surprise, revision volatility and conventional benchmark are not yet built.")


### Setup 5: Add the six-month uncertainty benchmark

In [ ]:
MPU_URL = "https://www.frbsf.org/wp-content/uploads/market-based-monetary-policy-uncertainty-data.xlsx"
mpu_bytes = download("mpu", MPU_URL)
mpu_book = pd.ExcelFile(io.BytesIO(mpu_bytes))
assert {"readme", "mpu"} <= set(mpu_book.sheet_names)
mpu = pd.read_excel(mpu_book, sheet_name="mpu")
assert {"Date", "6 months"} <= set(mpu.columns), "Benchmark schema changed"
mpu["Date"] = pd.to_datetime(mpu["Date"], errors="raise")
mpu["6 months"] = pd.to_numeric(mpu["6 months"], errors="raise")
assert mpu["Date"].is_unique and mpu["Date"].notna().all()
assert mpu["6 months"].dropna().ge(0).all()

# The source is already a standard deviation, in percentage points.
# Multiply by 100 for bps; do NOT take another square root.
mpu["mpu6m_bps"] = mpu["6 months"] * 100
enriched = dataset.merge(
    mpu[["Date", "mpu6m_bps"]].rename(columns={"Date": "mpu_observation_date"}),
    left_on="observation_date", right_on="mpu_observation_date",
    how="left", validate="one_to_one"
)
assert len(enriched) == len(dataset)
assert enriched["mpu_observation_date"].dropna().lt(
    enriched.loc[enriched["mpu_observation_date"].notna(), "meeting_date"]
).all()
print("Latest benchmark observation:", mpu["Date"].max().date())
print("Exact previous-day benchmark matches:", enriched["mpu6m_bps"].notna().sum(),
      "of", len(enriched))
print("No backward-as-of substitution or interpolation applied.")
print("Benchmark covers six-month policy-path uncertainty, not immediate-decision uncertainty.")
print("Underlying derivatives switch from Eurodollar to SOFR in October 2022.")


### Setup 6: Build five daily revisions

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar
from dateutil.easter import easter

# Five daily revisions need SIX level observations for the SAME meeting.
# In this sample, check that the selected weekdays contain no federal
# holidays or Good Friday. A future flagged window needs calendar review.
years = range(int(enriched["meeting_date"].dt.year.min()) - 1,
              int(enriched["meeting_date"].dt.year.max()) + 1)
closed_dates = USFederalHolidayCalendar().holidays(
    start=f"{min(years)}-01-01", end=f"{max(years)}-12-31"
).union(pd.DatetimeIndex([
    pd.Timestamp(easter(year)) - pd.Timedelta(days=2) for year in years
]))
revision_rows = []
window_rows = []
indexed = moments.set_index(["event_key", "date"]).sort_index()
for _, event in enriched.iterrows():
    dates = pd.bdate_range(end=event["observation_date"], periods=6)
    holiday_flag = bool(dates.isin(closed_dates).any())
    key = pd.MultiIndex.from_product(
        [[event["event_key"]], dates], names=["event_key", "date"]
    )
    window = indexed.reindex(key)
    mean_levels = window["mean"].to_numpy(dtype=float) * 100
    complete = bool(np.isfinite(mean_levels).all()) and not holiday_flag
    changes = np.diff(mean_levels) if complete else np.full(5, np.nan)
    rv5 = float(np.sqrt(np.sum(changes ** 2))) if complete else np.nan
    mean_abs = float(np.mean(np.abs(changes))) if complete else np.nan
    zero_volume_days = int(window["daily_volume"].eq(0).sum())
    revision_rows.append({
        "meeting_date": event["meeting_date"],
        "rv5_bps": rv5,
        "mean_abs_daily_revision_bps": mean_abs,
        "revision_level_observations": int(np.isfinite(mean_levels).sum()),
        "revision_window_complete": complete,
        "revision_calendar_review": holiday_flag,
        "window_zero_volume_days": zero_volume_days
    })
    for position, date in enumerate(dates):
        window_rows.append({
            "meeting_date": event["meeting_date"], "event_key": event["event_key"],
            "observation_date": date, "authors_adjusted_mean_bps": mean_levels[position],
            "change_from_previous_bps": changes[position-1] if position else np.nan,
            "authors_daily_volume": window["daily_volume"].iloc[position],
            "window_complete": complete
        })
revision_audit = pd.DataFrame(revision_rows)
revision_windows = pd.DataFrame(window_rows)
assert revision_windows["observation_date"].lt(revision_windows["meeting_date"]).all()
assert not revision_windows.duplicated(
    ["meeting_date", "observation_date"]
).any()
enriched = enriched.merge(revision_audit, on="meeting_date",
                          how="left", validate="one_to_one")
assert len(enriched) == len(dataset)
print("Complete five-revision windows:",
      int(enriched["revision_window_complete"].sum()), "of", len(enriched))
print("Calendar-review windows:", int(enriched["revision_calendar_review"].sum()))
print("Windows with zero published daily volume:",
      int(enriched["window_zero_volume_days"].gt(0).sum()))
print("Missing levels remain missing; no partial-window RV or zero filling.")
print("Published forward-filled levels may still conceal stale underlying strikes.")


### Setup 7: Fix model-specific samples

In [ ]:
# Fix the specifications before inspecting regressions.
# A: absolute yield move ~ decision sigma
# B0/B1: baseline alone versus baseline + decision sigma, on IDENTICAL rows
# P0/P1: baseline alone versus baseline + revision RV, estimated in parallel
# Do not put sigma, RV and many additional controls into a kitchen-sink model.
enriched["sample_A"] = np.isfinite(
    enriched[["abs_ust2y_bps", "kalshi_sigma_bps"]]
).all(axis=1)
enriched["sample_B"] = enriched["sample_A"] & np.isfinite(enriched["mpu6m_bps"])
enriched["sample_revision"] = (
    np.isfinite(enriched[["abs_ust2y_bps", "mpu6m_bps", "rv5_bps"]]).all(axis=1)
    & enriched["revision_window_complete"]
)
print(enriched[[
    "meeting_date", "kalshi_sigma_bps", "mpu6m_bps", "rv5_bps",
    "revision_level_observations", "sample_B", "sample_revision"
]].round(3).to_string(index=False))
print("\nFrozen candidate sample sizes:")
print("A, sigma only:", int(enriched["sample_A"].sum()))
print("B, baseline versus baseline + sigma:", int(enriched["sample_B"].sum()))
print("Parallel baseline versus baseline + RV:", int(enriched["sample_revision"].sum()))
print("\nPre-specified sensitivity checks:")
print("- Spearman correlation, HC3 inference and leave-one-event-out sensitivity")
print("- Zero-constructed-sigma exclusions as sensitivity only, not significance-based selection")
print("- Alternate statement/full-event windows and reasonable measurement conventions")
print("- Eurodollar/SOFR source transition is documented; avoid overfitting regime splits")
print("No regressions or significance searches have been run.")
print("Policy-surprise analysis remains separate; no realized surprise is an ex-ante predictor.")

OUTPUT = Path("research_outputs")
OUTPUT.mkdir(exist_ok=True)
enriched.to_csv(OUTPUT / "meeting_dataset_with_predictors.csv", index=False)
revision_windows.to_csv(OUTPUT / "revision_window_audit.csv", index=False)
revision_audit.to_csv(OUTPUT / "revision_completeness_audit.csv", index=False)
metadata = {
    "sources": source_receipts,
    "baseline": "Previous-day SF Fed 6-month market-based uncertainty; native pp * 100 = bps",
    "revision": "sqrt(sum of five squared revisions across six pre-announcement daily means)); means * 100 = bps",
    "timing": "Previous UTC day; weekday windows checked against federal holidays and Good Friday",
    "missingness": "Exact-date joins only; no additional forward filling; incomplete revision windows excluded only from revision models",
    "method_limits": [
        "Authors' daily Kalshi levels already carry forward stale prices and apply distribution repairs",
        "Daily aggregate volume does not identify exact age of every underlying strike",
        "Historical files are current-vintage processed data, not a verified real-time vintage archive",
        "Different market horizons and daily closing times limit direct comparability"
    ],
    "candidate_model_samples": {
        "A": int(enriched["sample_A"].sum()), "B": int(enriched["sample_B"].sum()),
        "revision_parallel": int(enriched["sample_revision"].sum())
    }
}
(OUTPUT / "predictor_provenance.json").write_text(json.dumps(metadata, indent=2))
print("\nSaved derived dataset, two revision audits and provenance under research_outputs/.")
print("Raw source files are not exported.")
print("Save to ResearchProject / main as 05_predictors_benchmark.ipynb.")


### Analysis 1: Load statistical tools and fix bootstrap settings

In [ ]:
import sys
import importlib.util
import subprocess
if importlib.util.find_spec("statsmodels") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "statsmodels"])
import statsmodels
import statsmodels.api as sm
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

BOOTSTRAP_REPS = 4999
RANDOM_SEED = 20261019
FIGURES = Path("research_outputs/figures")
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": "#242424", "text.color": "#242424",
    "figure.facecolor": "white", "axes.facecolor": "white"
})
BLUE, ORANGE, GREY = "#245B83", "#AF6427", "#666666"
print("Versions:", {"pandas": pd.__version__, "numpy": np.__version__,
                    "statsmodels": statsmodels.__version__})
print("Analysis scope fixed before fitting: A sigma; B baseline + sigma; parallel baseline + RV.")
print("Outcome: absolute statement-window two-year yield change in bps.")


### Analysis 2: Inspect Pearson and Spearman associations
Pearson measures linear association; Spearman measures rank association. Neither controls for the benchmark. Their descriptive p-values are not evidence of causality.

In [ ]:
correlation_rows = []
for predictor in ["kalshi_sigma_bps", "rv5_bps", "mpu6m_bps"]:
    data = enriched[["abs_ust2y_bps", predictor]].dropna()
    pearson = stats.pearsonr(data[predictor], data["abs_ust2y_bps"])
    spearman = stats.spearmanr(data[predictor], data["abs_ust2y_bps"])
    correlation_rows.append({
        "predictor": predictor, "n": len(data),
        "Pearson_r": pearson.statistic, "Pearson_p_unadjusted": pearson.pvalue,
        "Spearman_rho": spearman.statistic, "Spearman_p_unadjusted": spearman.pvalue
    })
correlations = pd.DataFrame(correlation_rows)
print(correlations.round(4).to_string(index=False))
print("These are unadjusted bivariate associations; p-values are descriptive and not multiplicity-adjusted.")
print("Spearman checks ordering without assuming a straight-line relationship.")


### Analysis 3: Estimate slopes, confidence intervals and incremental fit
An estimated coefficient of 0.4 would mean 5 bps more uncertainty is associated with 2 bps more absolute yield movement, holding the baseline fixed. Adding a predictor can improve ordinary R² mechanically; examine adjusted R², effect sizes, uncertainty and stability together. Intervals are approximate 95% HC3 t-based intervals, not prediction intervals.

In [ ]:
def fit_model(data, predictors, outcome="abs_ust2y_bps"):
    required = [outcome] + list(predictors)
    assert np.isfinite(data[required]).all().all(), "Inspect missing model inputs"
    X = sm.add_constant(data[list(predictors)].astype(float), has_constant="add")
    assert np.linalg.matrix_rank(X.to_numpy()) == X.shape[1], "Rank-deficient design"
    return sm.OLS(data[outcome].astype(float), X, missing="raise").fit(
        cov_type="HC3", use_t=True
    )

data_A = enriched.loc[enriched["sample_A"]].copy()
data_B = enriched.loc[enriched["sample_B"]].copy()
data_R = enriched.loc[enriched["sample_revision"]].copy()
models = {
    "A: sigma": fit_model(data_A, ["kalshi_sigma_bps"]),
    "B0: baseline": fit_model(data_B, ["mpu6m_bps"]),
    "B1: baseline + sigma": fit_model(data_B, ["mpu6m_bps", "kalshi_sigma_bps"]),
    "R0: baseline on RV sample": fit_model(data_R, ["mpu6m_bps"]),
    "R1: baseline + RV": fit_model(data_R, ["mpu6m_bps", "rv5_bps"])
}
coefficient_rows, fit_rows = [], []
for label, model in models.items():
    ci = model.conf_int(alpha=0.05)
    for term in model.params.index:
        coefficient_rows.append({
            "model": label, "term": term, "n": int(model.nobs),
            "coefficient": model.params[term], "HC3_se": model.bse[term],
            "CI95_low": ci.loc[term, 0], "CI95_high": ci.loc[term, 1],
            "HC3_t_p_unadjusted": model.pvalues[term]
        })
    fit_rows.append({
        "model": label, "n": int(model.nobs), "R2": model.rsquared,
        "adjusted_R2": model.rsquared_adj,
        "in_sample_RMSE_bps": np.sqrt(np.mean(model.resid ** 2))
    })
coefficients = pd.DataFrame(coefficient_rows)
model_fit = pd.DataFrame(fit_rows)
print("COEFFICIENTS: outcome bps per predictor bp; HC3 t-based 95% intervals")
print(coefficients.round(4).to_string(index=False))
print("\nIN-SAMPLE FIT")
print(model_fit.round(4).to_string(index=False))

comparisons = []
for label, reduced_key, full_key in [
    ("Add sigma to baseline", "B0: baseline", "B1: baseline + sigma"),
    ("Add RV to baseline", "R0: baseline on RV sample", "R1: baseline + RV")
]:
    reduced, full = models[reduced_key], models[full_key]
    assert np.array_equal(reduced.model.endog, full.model.endog)
    comparisons.append({
        "comparison": label, "n": int(full.nobs),
        "delta_R2": full.rsquared - reduced.rsquared,
        "delta_adjusted_R2": full.rsquared_adj - reduced.rsquared_adj,
        "partial_R2": (reduced.ssr - full.ssr) / reduced.ssr
    })
incremental_fit = pd.DataFrame(comparisons)
print("\nINCREMENTAL FIT ON IDENTICAL ROWS")
print(incremental_fit.round(4).to_string(index=False))
print("\nEffect-size illustration: a 5-bp higher predictor is associated with:")
for label, term in [("B1: baseline + sigma", "kalshi_sigma_bps"),
                    ("R1: baseline + RV", "rv5_bps")]:
    model = models[label]
    interval = model.conf_int().loc[term] * 5
    print(f"{label}: {model.params[term]*5:.3f} bps larger absolute yield move; "
          f"95% HC3 interval [{interval.iloc[0]:.3f}, {interval.iloc[1]:.3f}].")
print("In-sample fit is not out-of-sample forecasting performance.")


### Analysis 4: Read the scatterplots and adjusted relationship
Each dot is one meeting. The first plots are unadjusted. The residual plot removes the linear relationship with the baseline from both variables; its slope equals the adjusted sigma coefficient. These are explanatory visualizations, not a forecasting backtest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.0), sharey=True)
for ax, predictor, title, color in [
    (axes[0], "kalshi_sigma_bps", "Decision uncertainty", BLUE),
    (axes[1], "rv5_bps", "Expected-decision revisions", ORANGE)
]:
    data = enriched[["meeting_date", predictor, "abs_ust2y_bps"]].dropna()
    ax.scatter(data[predictor], data["abs_ust2y_bps"], s=43,
               color=color, edgecolor="white", linewidth=0.5, alpha=0.9)
    simple = fit_model(data, [predictor])
    grid = np.linspace(data[predictor].min(), data[predictor].max(), 100)
    ax.plot(grid, simple.params["const"] + simple.params[predictor] * grid,
            color=GREY, linewidth=1.5, label="Unadjusted OLS fit")
    ax.set(title=f"{title} (n={len(data)})", xlabel=f"{title} (bps)")
    ax.grid(axis="y", color="#E6E6E6", linewidth=0.7)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="lower right", fontsize=9)
axes[0].set_ylabel("Absolute two-year statement-window move (bps)")
fig.suptitle("Pre-announcement Kalshi measures and Treasury reactions", fontsize=14)
fig.text(0.5, 0.015,
         "Scheduled FOMC meetings, 2022–Sep 2026 | Published-method measures | Lines show association, not forecasts.",
         ha="center", fontsize=9, color=GREY)
fig.tight_layout(rect=(0, 0.055, 1, 0.95))
fig.savefig(FIGURES / "01_bivariate_scatterplots.png", dpi=160)
plt.show()

# Partial-regression view: remove the same baseline from both x and y.
Z = sm.add_constant(data_B[["mpu6m_bps"]], has_constant="add")
rx = sm.OLS(data_B["kalshi_sigma_bps"], Z).fit().resid
ry = sm.OLS(data_B["abs_ust2y_bps"], Z).fit().resid
beta = models["B1: baseline + sigma"].params["kalshi_sigma_bps"]
fig, ax = plt.subplots(figsize=(8.6, 5.4))
ax.scatter(rx, ry, s=45, color=BLUE, edgecolor="white", linewidth=0.5)
grid = np.linspace(rx.min(), rx.max(), 100)
ax.plot(grid, beta * grid, color=GREY, linewidth=1.5)
ax.axhline(0, color="#BBBBBB", linewidth=0.7)
ax.axvline(0, color="#BBBBBB", linewidth=0.7)
ax.set(title="Kalshi association after adjusting for the six-month benchmark",
       xlabel="Residual decision uncertainty (bps)",
       ylabel="Residual absolute yield reaction (bps)")
ax.grid(color="#EEEEEE", linewidth=0.6)
ax.set_axisbelow(True)
fig.text(0.5, 0.015, "Residuals can be negative. Adjustment removes a linear baseline association; it does not establish causality.",
         ha="center", fontsize=9, color=GREY)
fig.tight_layout(rect=(0, 0.055, 1, 1))
fig.savefig(FIGURES / "02_baseline_adjusted_scatter.png", dpi=160)
plt.show()
assert np.isclose(np.dot(rx, ry) / np.dot(rx, rx), beta), "Partial-regression slope mismatch"


### Analysis 5: Inspect individual-event influence and fixed sensitivity checks
Leave-one-out removes each meeting once. It is a sensitivity check, not out-of-sample evaluation. Constructed-zero exclusions and inactive-window exclusions remain secondary. The full monetary-event window includes the press conference and can change the economic interpretation.

In [ ]:
loo_rows = []
for label, data, predictors, term in [
    ("A: sigma", data_A, ["kalshi_sigma_bps"], "kalshi_sigma_bps"),
    ("B1: baseline + sigma", data_B, ["mpu6m_bps", "kalshi_sigma_bps"], "kalshi_sigma_bps"),
    ("R1: baseline + RV", data_R, ["mpu6m_bps", "rv5_bps"], "rv5_bps")
]:
    for index, event in data.iterrows():
        fit = fit_model(data.drop(index=index), predictors)
        ci = fit.conf_int().loc[term]
        loo_rows.append({
            "model": label, "omitted_meeting": event["meeting_date"],
            "coefficient": fit.params[term], "CI95_low": ci.iloc[0],
            "CI95_high": ci.iloc[1], "HC3_p_unadjusted": fit.pvalues[term]
        })
leave_one_out = pd.DataFrame(loo_rows)
print("LEAVE-ONE-EVENT-OUT coefficient ranges")
print(leave_one_out.groupby("model")["coefficient"].agg(["min", "max"]).round(4).to_string())

primary = models["B1: baseline + sigma"]
influence = primary.get_influence()
influence_table = pd.DataFrame({
    "meeting_date": data_B["meeting_date"].to_numpy(),
    "leverage": influence.hat_matrix_diag,
    "Cooks_distance": influence.cooks_distance[0]
}).sort_values("Cooks_distance", ascending=False)
print("\nLargest Cook's distances (diagnostic rankings, not deletion rules)")
print(influence_table.head(5).round(4).to_string(index=False))

fig, axes = plt.subplots(2, 1, figsize=(10.2, 7.8), sharex=True)
for ax, label, color in [
    (axes[0], "B1: baseline + sigma", BLUE),
    (axes[1], "R1: baseline + RV", ORANGE)
]:
    part = leave_one_out.loc[leave_one_out["model"].eq(label)].sort_values("omitted_meeting")
    term = "rv5_bps" if "RV" in label else "kalshi_sigma_bps"
    full_beta = models[label].params[term]
    ax.scatter(part["omitted_meeting"], part["coefficient"], color=color, s=30)
    ax.axhline(full_beta, color=GREY, linestyle="--", label="Full-sample coefficient")
    ax.axhline(0, color="#BBBBBB", linewidth=0.8)
    ax.set(title=label, ylabel="Coefficient after omission")
    ax.grid(axis="y", color="#E6E6E6")
    ax.legend(frameon=False, fontsize=9)
ticks = pd.DatetimeIndex([
    data_B["meeting_date"].min(), pd.Timestamp("2023-01-01"),
    pd.Timestamp("2024-01-01"), pd.Timestamp("2025-01-01"),
    pd.Timestamp("2026-01-01"), data_B["meeting_date"].max()
]).unique()
axes[-1].set_xticks(ticks)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
axes[-1].set_xlabel("Meeting omitted from the regression")
fig.suptitle("Influence of individual FOMC meetings", fontsize=14)
fig.text(0.5, 0.015, "Dots are sensitivity estimates, not independent observations or confidence intervals.",
         ha="center", fontsize=9, color=GREY)
fig.tight_layout(rect=(0, 0.045, 1, 0.95))
fig.savefig(FIGURES / "03_leave_one_out.png", dpi=160)
plt.show()

sensitivity_rows = []
def record_sensitivity(label, data, predictors, term, outcome="abs_ust2y_bps"):
    fit = fit_model(data, predictors, outcome)
    ci = fit.conf_int().loc[term]
    sensitivity_rows.append({
        "check": label, "n": len(data), "coefficient": fit.params[term],
        "CI95_low": ci.iloc[0], "CI95_high": ci.iloc[1],
        "HC3_p_unadjusted": fit.pvalues[term]
    })
record_sensitivity("B1 excluding constructed zero sigma",
                   data_B.loc[data_B["kalshi_sigma_bps"].gt(0)],
                   ["mpu6m_bps", "kalshi_sigma_bps"], "kalshi_sigma_bps")
record_sensitivity("R1 excluding zero-volume-day windows",
                   data_R.loc[data_R["window_zero_volume_days"].eq(0)],
                   ["mpu6m_bps", "rv5_bps"], "rv5_bps")
broad = pd.read_excel(workbook, sheet_name="Monetary Events")
assert {"Date", "UST2Y"} <= set(broad.columns)
broad["Date"] = pd.to_datetime(broad["Date"], errors="raise")
assert broad["Date"].is_unique
broad["abs_full_event_bps"] = pd.to_numeric(broad["UST2Y"], errors="raise").abs() * 100
broad_data = data_B.merge(
    broad[["Date", "abs_full_event_bps"]], left_on="meeting_date",
    right_on="Date", how="left", validate="one_to_one"
).dropna(subset=["abs_full_event_bps"])
record_sensitivity("B1 broader statement + press-conference outcome",
                   broad_data, ["mpu6m_bps", "kalshi_sigma_bps"],
                   "kalshi_sigma_bps", outcome="abs_full_event_bps")
sensitivities = pd.DataFrame(sensitivity_rows)
print("\nPRE-SPECIFIED SENSITIVITIES")
print(sensitivities.round(4).to_string(index=False))
print("All original observations remain in the primary analysis.")


### Analysis 6: Approximate restricted wild-bootstrap sensitivity
Under a zero added-predictor coefficient, retain the baseline fitted relationship and randomly flip leverage-adjusted residual signs. Compare absolute HC3 t-statistics. Independent event errors and an adequate conditional-mean model remain assumptions; there is no exact finite-sample guarantee. Use 4,999 draws and fixed seeds. Holm p-values adjust the two planned incremental predictor tests only.

In [ ]:
# Restricted wild bootstrap sensitivity under H0: added predictor coefficient = 0.
# Retain the baseline and fixed designs. Flip HC2-adjusted reduced-model residuals.
# Studentize each draw with HC3. This is approximate, not an exact finite-sample test,
# and it still assumes independent event errors and a suitable conditional-mean model.
def wild_bootstrap_added_predictor(data, predictor, reps, seed):
    X0 = sm.add_constant(data[["mpu6m_bps"]], has_constant="add").to_numpy()
    X1 = sm.add_constant(data[["mpu6m_bps", predictor]], has_constant="add").to_numpy()
    y = data["abs_ust2y_bps"].to_numpy()
    reduced = sm.OLS(y, X0).fit()
    full = sm.OLS(y, X1).fit(cov_type="HC3", use_t=True)
    leverage0 = reduced.get_influence().hat_matrix_diag
    adjusted_residuals = reduced.resid / np.sqrt(1 - leverage0)
    observed_t = abs(full.tvalues[-1])
    rng = np.random.default_rng(seed)
    exceed = 0
    for _ in range(reps):
        signs = rng.choice([-1.0, 1.0], size=len(y))
        y_star = reduced.fittedvalues + adjusted_residuals * signs
        fit = sm.OLS(y_star, X1).fit(cov_type="HC3", use_t=True)
        exceed += abs(fit.tvalues[-1]) >= observed_t
    p = (exceed + 1) / (reps + 1)
    return {
        "added_predictor": predictor, "n": len(data), "draws": reps,
        "seed": seed, "observed_abs_HC3_t": observed_t,
        "wild_bootstrap_p_two_sided": p,
        "Monte_Carlo_SE_approx": np.sqrt(p * (1 - p) / reps)
    }

bootstrap_results = pd.DataFrame([
    wild_bootstrap_added_predictor(data_B, "kalshi_sigma_bps", BOOTSTRAP_REPS, RANDOM_SEED),
    wild_bootstrap_added_predictor(data_R, "rv5_bps", BOOTSTRAP_REPS, RANDOM_SEED + 1)
])
# Two planned incremental predictor tests: show a simple conservative family-wise correction.
from statsmodels.stats.multitest import multipletests
bootstrap_results["Holm_p_two_incremental_tests"] = multipletests(
    bootstrap_results["wild_bootstrap_p_two_sided"], method="holm"
)[1]
print(bootstrap_results.round(5).to_string(index=False))
print("Wild-bootstrap p-values are approximate robustness checks, not proof of a model.")
print("Holm adjustment covers these two incremental tests, not all exploratory checks.")


### Analysis 7: Save reproducible results
Figures and derived results stay in the Colab runtime unless explicitly downloaded. Saving the notebook preserves its displayed outputs. Bootstrap draws and source hashes are recorded.

In [ ]:
tables = {
    "correlations": correlations, "coefficients_HC3": coefficients,
    "model_fit": model_fit, "incremental_fit": incremental_fit,
    "leave_one_event_out": leave_one_out, "influence": influence_table,
    "sensitivities": sensitivities, "wild_bootstrap": bootstrap_results
}
for name, table in tables.items():
    table.to_csv(OUTPUT / f"analysis_{name}.csv", index=False)
enriched.to_csv(OUTPUT / "analysis_input_dataset.csv", index=False)
manifest = {
    "source_receipts": source_receipts,
    "software": {"pandas": pd.__version__, "numpy": np.__version__,
                 "statsmodels": statsmodels.__version__},
    "bootstrap_draws": BOOTSTRAP_REPS, "bootstrap_seed": RANDOM_SEED,
    "inference": "OLS HC3 t-based intervals; approximate restricted wild bootstrap",
    "models": {label: {"n": int(m.nobs), "predictors": m.model.exog_names}
               for label, m in models.items()},
    "claim_scope": "Historical association and incremental in-sample explanatory value only",
    "remaining_work": [
        "Supporting realized target-rate surprise analysis",
        "Alternative uncertainty and measurement conventions",
        "Source construction and current-vintage sensitivity",
        "Final interpretation and research write-up"
    ]
}
(OUTPUT / "analysis_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved eight analysis tables, the input dataset, manifest and three figures.")
print("Read in order: scatterplots → coefficients/intervals → incremental fit → influence → robustness.")
print("A positive slope with an interval spanning zero is an imprecise estimate, not proof of no relationship.")
print("An interval excluding zero still does not establish causality or forecasting ability.")
print("Do not choose the final specification by searching for a smaller p-value.")
print("Save to ResearchProject / main as 06_initial_analysis.ipynb.")
